# Modulo 6 · Lezione 4 — Statistica descrittiva

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Misure di centralità: media, mediana, moda |
| 2 | Misure di dispersione: varianza, deviazione standard, range |
| 3 | Percentili, quartili e outlier |
| 4 | Forma della distribuzione: asimmetria e curtosi |
| 5 | Correlazione |
| 6 | Visualizzazione statistica |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print("Librerie pronte")

---
## Motivazione

```
Prima di addestrare qualsiasi modello di IA:

1. CAPIRE i dati — che range hanno? ci sono outlier? sono sbilanciati?
2. PULIRE i dati — individuare errori, valori impossibili
3. PREPARARE i dati — normalizzare, centrare, trasformare

La statistica descrittiva è il linguaggio per fare questa analisi.
```

In [ ]:
# Dataset: 50 studenti, età, voti e ore di studio
np.random.seed(42)
n = 50

edades  = np.random.normal(22, 3, n).clip(18, 35).round().astype(int)
horas   = np.random.normal(5, 1.5, n).clip(0.5, 10).round(1)
# Voto correlato con le ore + rumore
notas   = (horas * 0.8 + np.random.normal(0, 1, n) + 2).clip(1, 10).round(1)
# Aggiungiamo un outlier
notas[0] = 0.5
horas[1] = 0.2

print("Dataset di 50 studenti")
print(f"Età     (primi 10): {edades[:10]}")
print(f"Ore     (primi 10): {horas[:10]}")
print(f"Voti    (primi 10): {notas[:10]}")

---
## 1. Misure di centralità

```
MEDIA:   somma di tutti i valori / n
  ✓ Usa tutti i dati
  ✗ Molto sensibile agli outlier

MEDIANA: valore centrale quando ordiniamo dal più piccolo al più grande
  ✓ Robusta rispetto agli outlier
  ✓ Migliore per distribuzioni asimmetriche

MODA:    valore che compare con maggiore frequenza
  ✓ Unica metrica applicabile a dati categorici

Esempio: stipendi in una piccola azienda
  [25000, 26000, 27000, 28000, 250000] (il CEO)
  Media:  71200 ← distorta dal CEO
  Mediana: 27000 ← più rappresentativa
```

In [ ]:
print("=== Voti dei 50 studenti ===")
media   = np.mean(notas)
mediana = np.median(notas)
moda_r  = stats.mode(notas, keepdims=True)

print(f"Media:   {media:.4f}")
print(f"Mediana: {mediana:.4f}")
print(f"Moda:    {moda_r.mode[0]} (compare {moda_r.count[0]} volte)")

print("\n=== Effetto di un outlier ===")
notas_con_outlier = np.append(notas, [0.5, 0.3, 0.1])  # 3 outlier estremi
print(f"Senza outlier estremi: media={media:.2f}, mediana={mediana:.2f}")
print(f"Con outlier estremi: media={np.mean(notas_con_outlier):.2f}, mediana={np.median(notas_con_outlier):.2f}")
print("→ La mediana cambia appena; la media si sposta notevolmente")

# Visualizzazione
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(notas, bins=15, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(media,   color='red',    ls='-',  lw=2, label=f'Media = {media:.2f}')
ax.axvline(mediana, color='orange', ls='--', lw=2, label=f'Mediana = {mediana:.2f}')
ax.set_xlabel('Voto'); ax.set_ylabel('Frequenza')
ax.set_title('Distribuzione dei voti')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 2. Misure di dispersione

```
RANGE:                    max - min
VARIANZA (σ²):            media degli scarti al quadrato
DEVIAZIONE STANDARD (σ):  √varianza — stesse unità dei dati
ERRORE STANDARD:          σ / √n — precisione della media stimata

Popolazione vs Campione:
  Varianza della popolazione: σ² = Σ(xᵢ - μ)² / N      (ddof=0)
  Varianza campionaria:       s² = Σ(xᵢ - x̄)² / (N-1)  (ddof=1) ← usare di default

Perché N-1? Correzione di Bessel: la varianza campionaria sottostima
la varianza reale se usiamo N. N-1 corregge questo bias.
```

In [ ]:
print("=== Dispersione dei voti ===")
rango    = np.max(notas) - np.min(notas)
var_pob  = np.var(notas, ddof=0)   # varianza della popolazione
var_mues = np.var(notas, ddof=1)   # varianza campionaria
std_pob  = np.std(notas, ddof=0)
std_mues = np.std(notas, ddof=1)
error_std= std_mues / np.sqrt(len(notas))

print(f"Minimo:               {np.min(notas):.1f}")
print(f"Massimo:              {np.max(notas):.1f}")
print(f"Range:                {rango:.1f}")
print(f"Varianza (N):         {var_pob:.4f}")
print(f"Varianza (N-1):       {var_mues:.4f}")
print(f"Dev. standard (N):    {std_pob:.4f}")
print(f"Dev. standard (N-1):  {std_mues:.4f}")
print(f"Errore standard:      {error_std:.4f}")
print(f"\nInterpretazione: Il voto tipico varia di ±{std_mues:.2f} punti rispetto alla media ({media:.2f})")

# Confronto tra due gruppi con stessa media, diversa dispersione
print("\n=== Due classi con stessa media, diversa dispersione ===")
clase_A = np.array([6.5, 7.0, 6.8, 7.2, 7.0, 6.9, 7.1, 7.0, 6.7, 7.3])
clase_B = np.array([3.0, 4.5, 5.5, 7.0, 8.5, 9.5, 7.5, 6.0, 5.0, 8.0])
print(f"Classe A: media={clase_A.mean():.2f}, std={clase_A.std():.2f} → voti molto omogenei")
print(f"Classe B: media={clase_B.mean():.2f}, std={clase_B.std():.2f} → voti molto dispersi")

---
## 3. Percentili, quartili e outlier

In [ ]:
print("=== Percentili ===")
for p in [10, 25, 50, 75, 90, 95, 99]:
    val = np.percentile(notas, p)
    print(f"  P{p:2d}: {val:.2f}")

Q1  = np.percentile(notas, 25)
Q3  = np.percentile(notas, 75)
IQR = Q3 - Q1

limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

print(f"\n=== Quartili e outlier ===")
print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = Q3 - Q1 = {IQR:.2f}")
print(f"Limiti: [{limite_inf:.2f}, {limite_sup:.2f}]")

outliers = notas[(notas < limite_inf) | (notas > limite_sup)]
print(f"Outlier rilevati: {outliers}")

# Boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(notas, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', lw=2),
                flierprops=dict(marker='o', color='red', ms=8))
axes[0].set_ylabel('Voto'); axes[0].set_title('Boxplot dei voti')
axes[0].set_xticklabels(['Studenti'])

axes[1].boxplot([notas, clase_A, clase_B], vert=True, patch_artist=True,
                labels=['Generale', 'Classe A', 'Classe B'],
                boxprops=dict(facecolor='lightgreen'),
                medianprops=dict(color='red', lw=2))
axes[1].set_ylabel('Voto'); axes[1].set_title('Confronto tra gruppi')

plt.tight_layout(); plt.show()

print("\n=== Descrizione completa (numpy) ===")
resumen = {
    'n':      len(notas),
    'media':  round(np.mean(notas), 3),
    'std':    round(np.std(notas, ddof=1), 3),
    'min':    np.min(notas),
    'Q1':     round(Q1, 3),
    'mediana':round(np.median(notas), 3),
    'Q3':     round(Q3, 3),
    'max':    np.max(notas),
}
for k, v in resumen.items():
    print(f"  {k:10}: {v}")

---
## 4. Forma della distribuzione

In [ ]:
print("=== Asimmetria (skewness) ===")
print("  skewness = 0  → distribuzione simmetrica")
print("  skewness > 0  → coda a destra (media > mediana)")
print("  skewness < 0  → coda a sinistra (media < mediana)")

asimetria = stats.skew(notas)
curtosis  = stats.kurtosis(notas)
print(f"\nVoti: asimmetria={asimetria:.4f}, curtosi={curtosis:.4f}")

# Distribuzioni con diversa asimmetria
np.random.seed(42)
simetrica    = np.random.normal(0, 1, 1000)
cola_derecha = np.random.exponential(1, 1000)          # coda a destra
cola_izquierda = -np.random.exponential(1, 1000)       # coda a sinistra

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, datos, titulo in zip(axes,
    [simetrica, cola_derecha, cola_izquierda],
    ['Simmetrica', 'Coda destra (+)', 'Coda sinistra (-)']):
    ax.hist(datos, bins=40, density=True, alpha=0.7, color='steelblue')
    ax.axvline(np.mean(datos),   color='red',    ls='-', lw=2, label='Media')
    ax.axvline(np.median(datos), color='orange', ls='--',lw=2, label='Mediana')
    skew = stats.skew(datos)
    ax.set_title(f"{titulo}\nskewness={skew:.2f}")
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## 5. Correlazione

```
La correlazione di Pearson misura la relazione lineare tra due variabili.

r ∈ [-1, 1]
  r = 1   → relazione lineare positiva perfetta
  r = -1  → relazione lineare negativa perfetta
  r = 0   → nessuna relazione lineare

IMPORTANTE: correlazione ≠ causalità
  I gelati venduti correlano con gli annegamenti → causa: il caldo
  La cicogna correla con le nascite → entrambe dipendono dalla ruralità
```

In [ ]:
r_pearson, p_valor = stats.pearsonr(horas, notas)
print(f"Correlazione ore↔voti: r={r_pearson:.4f}, p={p_valor:.4f}")
print(f"Significativa (p<0.05): {p_valor < 0.05}")

# Matrice di correlazione
datos_matriz = np.column_stack([edades, horas, notas])
nombres_vars = ['Età', 'Ore studio', 'Voto']
corr_matrix  = np.corrcoef(datos_matriz.T)

print("\nMatrice di correlazione:")
header = f"{'':15}" + "".join(f"{n:>15}" for n in nombres_vars)
print(header)
for i, nombre in enumerate(nombres_vars):
    fila = f"{nombre:15}" + "".join(f"{corr_matrix[i,j]:>15.4f}" for j in range(3))
    print(fila)

# Visualizzazione
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter ore vs voti
axes[0].scatter(horas, notas, alpha=0.6, color='steelblue')
m, b = np.polyfit(horas, notas, 1)
x_line = np.linspace(horas.min(), horas.max(), 100)
axes[0].plot(x_line, m*x_line + b, 'r-', lw=2)
axes[0].set_xlabel('Ore di studio'); axes[0].set_ylabel('Voto')
axes[0].set_title(f'Ore vs Voto (r={r_pearson:.3f})')
axes[0].grid(True, alpha=0.3)

# Heatmap di correlazione
im = axes[1].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(3)); axes[1].set_yticks(range(3))
axes[1].set_xticklabels(nombres_vars, rotation=45, ha='right')
axes[1].set_yticklabels(nombres_vars)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f"{corr_matrix[i,j]:.2f}", ha='center', va='center', fontsize=11)
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Matrice di correlazione')

plt.tight_layout(); plt.show()

# La correlazione non implica causalità
print("\n=== Correlazioni ingannevoli ===")
np.random.seed(42)
x_ruido = np.random.randn(50)
y_cuad  = x_ruido**2 + np.random.randn(50) * 0.1
r_cuad, _ = stats.pearsonr(x_ruido, y_cuad)
print(f"Relazione quadratica y=x²: r_pearson={r_cuad:.4f} (quasi 0! ma la relazione è chiara)")
print("→ Pearson rileva solo relazioni LINEARI")

---
## Pratica

### Esercizio 1 — Analisi degli stipendi

**Consegna:** Dato questo dataset di stipendi annuali (€):

```python
salarios = [28000, 32000, 35000, 31000, 29000, 38000, 34000,
            33000, 30000, 36000, 29500, 37000, 31500, 350000,
            32500, 28500, 39000, 33500, 30500, 36500]
```

1. Calcola media, mediana e moda
2. Calcola deviazione standard e IQR
3. Individua gli outlier con il metodo IQR
4. Confronta la media con e senza outlier
5. Qual è lo stipendio che supera l'80% del campione?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
from scipy import stats

salarios = np.array([28000, 32000, 35000, 31000, 29000, 38000, 34000,
                     33000, 30000, 36000, 29500, 37000, 31500, 350000,
                     32500, 28500, 39000, 33500, 30500, 36500])

# 1.
media   = np.mean(salarios)
mediana = np.median(salarios)
moda    = stats.mode(salarios, keepdims=True)
print(f"Media:   {media:,.0f}€")
print(f"Mediana: {mediana:,.0f}€")
print(f"Moda:    {moda.mode[0]:,.0f}€ (tutti unici → non c'è una moda chiara)")

# 2.
std = np.std(salarios, ddof=1)
Q1  = np.percentile(salarios, 25)
Q3  = np.percentile(salarios, 75)
IQR = Q3 - Q1
print(f"\nDev. standard: {std:,.0f}€")
print(f"IQR: {IQR:,.0f}€  (Q1={Q1:,.0f}, Q3={Q3:,.0f})")

# 3. Outlier
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
outliers = salarios[(salarios < lim_inf) | (salarios > lim_sup)]
print(f"\nLimiti: [{lim_inf:,.0f}, {lim_sup:,.0f}]")
print(f"Outlier: {outliers}")

# 4.
sin_outliers = salarios[(salarios >= lim_inf) & (salarios <= lim_sup)]
print(f"\nMedia CON outlier:    {np.mean(salarios):,.0f}€")
print(f"Media SENZA outlier:  {np.mean(sin_outliers):,.0f}€")
print(f"Differenza: {np.mean(salarios) - np.mean(sin_outliers):,.0f}€")

# 5.
p80 = np.percentile(salarios, 80)
print(f"\nPercentile 80: {p80:,.0f}€")
```
</details>

---
### Esercizio 2 — Confronto tra gruppi

**Consegna:** Tre modelli di ML producono questi tassi di errore in 20 esecuzioni:

```python
np.random.seed(0)
modelo_A = np.random.normal(0.15, 0.02, 20)
modelo_B = np.random.normal(0.13, 0.05, 20)
modelo_C = np.random.normal(0.14, 0.01, 20)
```

1. Per ogni modello: media, mediana, std, IQR
2. Qual è il modello migliore? Basta guardare la media?
3. Visualizza i tre modelli con boxplot e violin plot
4. Quale ha minore variabilità? Perché la variabilità è importante in produzione?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
modelo_A = np.random.normal(0.15, 0.02, 20)
modelo_B = np.random.normal(0.13, 0.05, 20)
modelo_C = np.random.normal(0.14, 0.01, 20)
modelos  = {'A': modelo_A, 'B': modelo_B, 'C': modelo_C}

# 1. Metriche
for nombre, m in modelos.items():
    Q1, Q3 = np.percentile(m, [25, 75])
    print(f"Modello {nombre}: media={m.mean():.4f}, std={m.std(ddof=1):.4f}, "
          f"mediana={np.median(m):.4f}, IQR={Q3-Q1:.4f}")

# 2. Analisi
print("\nB ha la media più bassa MA la maggiore variabilità (std=0.05)")
print("C ha media intermedia ma la minore variabilità (std=0.01)")
print("→ In produzione: C può essere più affidabile anche se non è il più preciso in media")

# 3. Visualizzazione
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

data = [modelo_A, modelo_B, modelo_C]
axes[0].boxplot(data, labels=['A','B','C'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', lw=2))
axes[0].set_title('Boxplot - Errore per modello'); axes[0].set_ylabel('Tasso di errore')
axes[0].grid(True, alpha=0.3)

axes[1].violinplot(data, positions=[1,2,3], showmedians=True)
axes[1].set_xticks([1,2,3]); axes[1].set_xticklabels(['A','B','C'])
axes[1].set_title('Violin plot'); axes[1].set_ylabel('Tasso di errore')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 3 — Normalizzazione

**Consegna:** Implementa e confronta tre metodi di normalizzazione:

```
Min-Max (scala [0,1]):           x' = (x - min) / (max - min)
Z-score (standard):              x' = (x - μ) / σ
Robusto (resistente agli outlier): x' = (x - mediana) / IQR
```

Applicali a `[2, 5, 3, 8, 100, 4, 6, 3, 7, 5]` e confronta l'effetto di ciascun metodo con e senza l'outlier (100).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np

datos = np.array([2, 5, 3, 8, 100, 4, 6, 3, 7, 5], dtype=float)

def min_max(x):  return (x - x.min()) / (x.max() - x.min())
def zscore(x):   return (x - x.mean()) / x.std(ddof=1)
def robusto(x):
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    return (x - med) / iqr

print("Con outlier (100):")
print(f"{'Valore':>8} {'Min-Max':>10} {'Z-score':>10} {'Robusto':>10}")
for orig, mm, zs, rob in zip(datos, min_max(datos), zscore(datos), robusto(datos)):
    print(f"{orig:>8.0f} {mm:>10.4f} {zs:>10.4f} {rob:>10.4f}")

datos_sin = datos[datos != 100]
print("\nSenza outlier:")
print(f"{'Valore':>8} {'Min-Max':>10} {'Z-score':>10} {'Robusto':>10}")
for orig, mm, zs, rob in zip(datos_sin, min_max(datos_sin), zscore(datos_sin), robusto(datos_sin)):
    print(f"{orig:>8.0f} {mm:>10.4f} {zs:>10.4f} {rob:>10.4f}")

print("\nConclusione:")
print("  Min-Max: molto influenzato dagli outlier (comprime il range normale)")
print("  Z-score: moderatamente influenzato")
print("  Robusto: quasi non cambia → ideale in presenza di outlier")
```
</details>

---
### Esercizio 4 — Analisi completa di un dataset

**Consegna:** Genera e analizza questo dataset di consumo energetico di edifici:

```python
np.random.seed(7)
n = 100
superficie  = np.random.normal(250, 80, n).clip(50, 600)
ocupantes   = np.random.poisson(15, n).clip(1, 50)
antiguedad  = np.random.uniform(0, 40, n)
consumo_kwh = (superficie * 1.2 + ocupantes * 200 + antiguedad * 50
               + np.random.normal(0, 2000, n)).clip(500)
```

1. Calcola le statistiche descrittive per tutte le variabili
2. Matrice di correlazione e heatmap
3. Quale variabile ha maggiore correlazione con il consumo?
4. Individua gli edifici con consumo anomalmente alto (outlier)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(7)
n = 100
superficie  = np.random.normal(250, 80, n).clip(50, 600)
ocupantes   = np.random.poisson(15, n).clip(1, 50).astype(float)
antiguedad  = np.random.uniform(0, 40, n)
consumo     = (superficie * 1.2 + ocupantes * 200 + antiguedad * 50
               + np.random.normal(0, 2000, n)).clip(500)

vars_dict = {'Superficie': superficie, 'Occupanti': ocupantes,
             'Anzianità': antiguedad, 'Consumo kWh': consumo}

# 1. Descrittive
print(f"{'Variabile':<15} {'Media':>10} {'Std':>10} {'Min':>10} {'Med':>10} {'Max':>10}")
for nombre, v in vars_dict.items():
    print(f"{nombre:<15} {v.mean():>10.1f} {v.std(ddof=1):>10.1f} {v.min():>10.1f} {np.median(v):>10.1f} {v.max():>10.1f}")

# 2. Correlazione
X = np.column_stack(list(vars_dict.values()))
C = np.corrcoef(X.T)
nombres = list(vars_dict.keys())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(4)); axes[0].set_yticks(range(4))
axes[0].set_xticklabels(nombres, rotation=45, ha='right')
axes[0].set_yticklabels(nombres)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f"{C[i,j]:.2f}", ha='center', va='center')
plt.colorbar(im, ax=axes[0]); axes[0].set_title('Correlazione')

axes[1].boxplot([(consumo - np.median(consumo)) / (np.percentile(consumo,75)-np.percentile(consumo,25))],
                vert=False, patch_artist=True, boxprops=dict(facecolor='lightblue'),
                flierprops=dict(marker='o', color='red', ms=8))
axes[1].set_title('Consumo normalizzato (outlier in rosso)'); axes[1].set_xlabel('Score robusto')
plt.tight_layout(); plt.show()

# 3. Maggiore correlazione con il consumo
idx_consumo = 3
for i, nombre in enumerate(nombres[:-1]):
    print(f"r({nombre}, Consumo) = {C[i, idx_consumo]:.4f}")

# 4. Outlier nel consumo
Q1, Q3 = np.percentile(consumo, [25, 75])
IQR = Q3 - Q1
outliers_idx = np.where((consumo > Q3 + 1.5*IQR))[0]
print(f"\nEdifici con consumo anomalo: {len(outliers_idx)}")
for i in outliers_idx:
    print(f"  Edificio {i}: {consumo[i]:.0f} kWh (sup={superficie[i]:.0f}m², occ={ocupantes[i]:.0f})")
```
</details>

---
### Esercizio 5 (sfida) — Dashboard statistico

**Consegna:** Implementa una funzione `resumen_estadistico(datos, nombre)` che:

1. Calcola tutte le metriche descrittive
2. Individua gli outlier
3. Classifica la distribuzione (simmetrica/asimmetrica sx/asimmetrica dx)
4. Genera un pannello di 4 grafici: istogramma+KDE, boxplot, QQ-plot, istogramma cumulato

Provala con i dati di consumo, voti e stipendi degli esercizi precedenti.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def resumen_estadistico(datos, nombre="Variable"):
    datos = np.array(datos, dtype=float)
    n = len(datos)
    media, mediana = np.mean(datos), np.median(datos)
    std   = np.std(datos, ddof=1)
    Q1, Q3 = np.percentile(datos, [25, 75])
    IQR   = Q3 - Q1
    skew  = stats.skew(datos)
    kurt  = stats.kurtosis(datos)
    _, p_norm = stats.shapiro(datos[:50] if n > 50 else datos)

    # Classificare la forma
    if abs(skew) < 0.5:   forma = "Approssimativamente simmetrica"
    elif skew > 0:         forma = "Asimmetrica destra (coda +)"
    else:                  forma = "Asimmetrica sinistra (coda -)"

    outliers = datos[(datos < Q1-1.5*IQR) | (datos > Q3+1.5*IQR)]

    print(f"{'='*50}")
    print(f" {nombre} (n={n})")
    print(f"{'='*50}")
    print(f" Media:     {media:.4f}  | Mediana: {mediana:.4f}")
    print(f" Std:       {std:.4f}  | IQR:     {IQR:.4f}")
    print(f" Min:       {datos.min():.4f}  | Max:     {datos.max():.4f}")
    print(f" Asimmetria:{skew:.4f}  | Curtosi: {kurt:.4f}")
    print(f" Forma:     {forma}")
    print(f" Normale (Shapiro p={p_norm:.4f}): {'Sì' if p_norm>0.05 else 'No'}")
    print(f" Outlier:   {len(outliers)}  {outliers[:5]}")

    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    fig.suptitle(nombre, fontsize=11)

    # Istogramma + KDE
    axes[0].hist(datos, bins='auto', density=True, alpha=0.6, color='steelblue')
    kde = stats.gaussian_kde(datos)
    x_kde = np.linspace(datos.min(), datos.max(), 200)
    axes[0].plot(x_kde, kde(x_kde), 'r-', lw=2)
    axes[0].axvline(media, color='orange', ls='--', lw=1.5, label='media')
    axes[0].axvline(mediana, color='green', ls=':', lw=1.5, label='mediana')
    axes[0].legend(fontsize=7); axes[0].set_title('Istogramma + KDE')

    # Boxplot
    axes[1].boxplot(datos, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', lw=2))
    axes[1].set_title('Boxplot')

    # QQ-plot
    stats.probplot(datos, plot=axes[2])
    axes[2].set_title('QQ-plot (vs Normale)')

    # CDF empirica
    x_sorted = np.sort(datos)
    cdf = np.arange(1, n+1) / n
    axes[3].step(x_sorted, cdf, where='post', color='steelblue', lw=2)
    axes[3].set_title('CDF empirica')
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

# Prova
np.random.seed(42)
resumen_estadistico(np.random.normal(7, 1.5, 50).clip(0, 10), "Voti")
```
</details>

---
## Riepilogo

```python
import numpy as np
from scipy import stats

x = np.array([...])

# Centralità
np.mean(x); np.median(x); stats.mode(x)

# Dispersione
np.std(x, ddof=1)          # deviazione standard campionaria
np.var(x, ddof=1)          # varianza campionaria
np.percentile(x, [25,75])  # Q1, Q3
# IQR = Q3 - Q1

# Outlier
outliers = x[(x < Q1 - 1.5*IQR) | (x > Q3 + 1.5*IQR)]

# Forma
stats.skew(x)              # asimmetria
stats.kurtosis(x)          # curtosi

# Correlazione
np.corrcoef(x, y)          # matrice di correlazione
stats.pearsonr(x, y)       # r + p-value
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 5** facciamo il salto verso l'inferenza statistica: come possiamo trarre conclusioni su una popolazione a partire da un campione?